# Hands-on session 7: Universal MLIPs and fine-tuning

**MS.49900 AI for Materials Science (in-class, 75 min)**

| Practice | What you do | Time |
|---|---|---|
| 1 | Compare universal MLIPs (uMLIP) on Li₆PS₅Cl | 20 min |
| 2 | Fine-tune a small model on 42 DFT structures | 25 min |
| 3 | Relax the structure and run NVT MD with your model | 15 min |

Run the cells in order. Cells marked `# You can change here` are blanks.

## Setup
Use a GPU runtime: **Runtime > Change runtime type > T4 GPU**.

In [ ]:
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU. Runtime > Change runtime type > T4 GPU, then run again from the top.')

In [ ]:
# Install SevenNet and download the course repository with the Li6PS5Cl DFT data (about 1 min)
!pip install -q sevenn==0.13.0

import os
if not os.path.exists('MS49900-AI4M'):
    !git clone -q https://github.com/kwongibaek/MS49900-AI4M.git

DATA = 'MS49900-AI4M/Data'
print(sorted(f for f in os.listdir(DATA) if f.startswith(('lpsc_', 'sevennet_'))))

In [ ]:
import time, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ase.io import read
from ase.units import bar
from sevenn.calculator import SevenNetCalculator

to_kbar = 1 / bar / 1000          # eV/Å³ -> kbar


def dft_labels(traj):
    """DFT energy (eV/atom), forces (eV/Å) and stress (kbar) stored in the data file."""
    E = np.array([a.get_potential_energy() / len(a) for a in traj])
    F = np.concatenate([a.get_forces().ravel() for a in traj])
    S = np.concatenate([a.get_stress().ravel() for a in traj]) * to_kbar
    return E, F, S


def predict(calc, traj):
    """MLIP energy, forces and stress for every structure, plus seconds per structure."""
    E, F, S = [], [], []
    warm = traj[0].copy(); warm.calc = calc; warm.get_forces()   # warm-up, not timed
    t0 = time.time()
    for atoms in traj:
        a = atoms.copy()          # copy keeps the DFT labels of the original
        a.calc = calc
        E.append(a.get_potential_energy() / len(a))
        F.append(a.get_forces().ravel())
        S.append(a.get_stress().ravel() * to_kbar)
    sec = (time.time() - t0) / len(traj)
    return np.array(E), np.concatenate(F), np.concatenate(S), sec


def free_gpu(calc=None):
    del calc
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))

def mae(y_true, y_pred):
    return np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))

def parity(ax, x, y, label, unit):
    ax.scatter(x, y, s=3, alpha=0.3)
    lo, hi = min(x.min(), y.min()), max(x.max(), y.max())
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1)
    ax.set_xlabel(f'DFT {label} ({unit})')
    ax.set_ylabel(f'MLIP {label} ({unit})')
    ax.set_aspect('equal')

## Practice 1. Compare universal MLIPs (zero-shot)

A **universal MLIP** is trained once on large DFT databases covering most of the periodic table. Larger models trained on more data rank higher on benchmarks such as [Matbench Discovery](https://matbench-discovery.materialsproject.org/). Do they also work better on *our* system?

| Model | Training data | Size |
|---|---|---|
| `7net-0` | MPtrj (1.6 M structures) | 0.8 M parameters |
| `7net-mf-ompa` | MPtrj + sAlex + OMat24 | larger, multi-fidelity |
| `7net-omni` | 15 open DFT datasets | largest, multi-task |

We test on 10 snapshots of a separate DFT MD run (`test_md.extxyz`).

In [ ]:
# Settings
N_TEST = 10
MODELS = {                      # label: (model name, extra options)
    '7net-0':       ('7net-0', {}),
    '7net-mf-ompa': ('7net-mf-ompa', {'modal': 'mpa'}),
    '7net-omni':    ('7net-omni', {'modal': 'mpa'}),
}

In [ ]:
test_all = read(f'{DATA}/lpsc_test_md.extxyz.gz', ':')
test = test_all[::len(test_all) // N_TEST][:N_TEST]
dft_E, dft_F, dft_S = dft_labels(test)
print(len(test), 'test structures,', len(test[0]), 'atoms each')

**Blank 1.** The energy offset from the pre-class hides how well each model ranks structures. Add the energy RMSE after removing the offset.

In [ ]:
results, preds = {}, {}

def add_result(label, calc):
    E, F, S, sec = predict(calc, test)
# You can change here------------------------------------------------------------
    e_shift = ...   # energy RMSE after removing each mean (see pre-class)
# -------------------------------------------------------------------------------
    preds[label] = (E, F, S)
    results[label] = {
        'Energy RMSE (eV/atom)': rmse(dft_E, E),
        'Energy RMSE, offset removed (meV/atom)': 1000 * e_shift,
        'Force RMSE (eV/Å)': rmse(dft_F, F),
        'Stress RMSE (kbar)': rmse(dft_S, S),
        'Time (s/structure)': sec,
    }

for label, (name, options) in MODELS.items():
    print('Running', label, '...')
    calc = SevenNetCalculator(name, **options)
    add_result(label, calc)
    free_gpu(calc)

pd.DataFrame(results).T.round(3)

In [ ]:
def plot_parity(labels):
    fig, axes = plt.subplots(len(labels), 3, figsize=(12, 3.8 * len(labels)), squeeze=False)
    for row, label in zip(axes, labels):
        E, F, S = preds[label]
        parity(row[0], dft_E - dft_E.mean(), E - E.mean(), 'rel. energy', 'eV/atom')
        parity(row[1], dft_F, F, 'force', 'eV/Å')
        parity(row[2], dft_S, S, 'stress', 'kbar')
        row[0].set_title(label, loc='left', fontweight='bold')
    plt.tight_layout(); plt.show()

plot_parity(list(MODELS))

**Check point.** Which model gives the lowest force error? How much slower is it than `7net-0`?

If a model runs out of GPU memory, restart the runtime and remove it from `MODELS`.

## Practice 2. Fine-tune a small model

Instead of a bigger model, we adapt a small pretrained model (`7net-0-small`) to our DFT setting with a few structures, in two steps.

1. **Shift only**: fix the constant energy offset by moving the reference energies. No training, forces and stress do not change.
2. **Fine-tune**: retrain all weights from the pretrained values.

- Data: 600 K + 1200 K snapshots, 25% for training (42 structures), 10 for validation
- Train: 10 epochs, Adam, learning rate 0.004 decreasing linearly

In [ ]:
import sevenn.util as util
from copy import deepcopy
from torch_geometric.loader import DataLoader
from sevenn.train.graph_dataset import SevenNetGraphDataset
from sevenn.train.trainer import Trainer
from sevenn.error_recorder import ErrorRecorder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(0)

# Settings
TRAIN_FRACTION = 0.25
N_VALID = 10
EPOCHS = 10
BATCH_SIZE = 4
TRAIN_SETTINGS = {
    'device': device,
    'is_ddp': False,
    'optimizer': 'adam',
    'optim_param': {'lr': 0.004},
    'scheduler': 'linearlr',
    'scheduler_param': {'start_factor': 1.0, 'total_iters': EPOCHS, 'end_factor': 0.0001},
    'loss': 'huber',
    'loss_param': {'delta': 0.01},
    'force_loss_weight': 0.1,
    'stress_loss_weight': 0.01,
}

model, config = util.model_from_checkpoint(f'{DATA}/sevennet_0_small_checkpoint.pth')
config.update(TRAIN_SETTINGS)

dataset = SevenNetGraphDataset(
    cutoff=config['cutoff'], root='sevenn_data', drop_info=False,
    files=[f'{DATA}/lpsc_600K.extxyz.gz', f'{DATA}/lpsc_1200K.extxyz.gz'],
)
dataset = dataset.shuffle()
n_train = int(len(dataset) * TRAIN_FRACTION)
train_loader = DataLoader(dataset[:n_train], batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(dataset[n_train:n_train + N_VALID], batch_size=BATCH_SIZE)
print(f'{len(dataset)} structures: {n_train} train, {N_VALID} validation')

### Step 1. Shift only

Each SevenNet model ends with a reference energy per element. Moving all of them by the same amount shifts the total energy by a constant and leaves forces and stress unchanged.

**Blank 2.** Compute the offset (eV/atom) between DFT and `7net-0-small` on a few training structures.

*(All our structures have the same composition, so one offset for every element is enough. For datasets with different compositions, fit one offset per element by linear regression, as in the SevenNet tutorial.)*

In [ ]:
train_atoms = read(f'{DATA}/lpsc_600K.extxyz.gz', '::20') + read(f'{DATA}/lpsc_1200K.extxyz.gz', '::20')
calc = SevenNetCalculator(f'{DATA}/sevennet_0_small_checkpoint.pth')
E_pred, _, _, _ = predict(calc, train_atoms)
free_gpu(calc)
E_dft = np.array([a.get_potential_energy() / len(a) for a in train_atoms])
# You can change here------------------------------------------------------------
offset = ...   # mean of (DFT - model) energy per atom, eV/atom
# -------------------------------------------------------------------------------
print(f'offset = {offset:.3f} eV/atom from {len(train_atoms)} structures')

shift_model, shift_config = util.model_from_checkpoint(f'{DATA}/sevennet_0_small_checkpoint.pth')
with torch.no_grad():
    shift_model._modules['rescale_atomic_energy'].shift += offset    # same offset for every element
shift_config.update(TRAIN_SETTINGS)
Trainer.from_config(shift_model, shift_config).write_checkpoint('checkpoint_shift.pth', config=shift_config, epoch=0)
free_gpu()

### Step 2. Fine-tune

In [ ]:
trainer = Trainer.from_config(model, config)
train_rec = ErrorRecorder.from_config(config)
valid_rec = deepcopy(train_rec)

history = []
for epoch in range(1, EPOCHS + 1):
    trainer.run_one_epoch(train_loader, is_train=True, error_recorder=train_rec)
    trainer.run_one_epoch(valid_loader, is_train=False, error_recorder=valid_rec)
    trainer.scheduler_step()
    tr, va = train_rec.epoch_forward(), valid_rec.epoch_forward()
    history.append(va)
    print(f"epoch {epoch:2d} | valid MAE  E {va['Energy_MAE (eV/atom)']:.4f} eV/atom"
          f"  F {va['Force_MAE (eV/Å)']:.4f} eV/Å  S {va['Stress_MAE (kbar)']:.3f} kbar")

trainer.write_checkpoint('checkpoint_fine_tuned.pth', config=config, epoch=EPOCHS)
free_gpu()

**Blank 3.** Load your fine-tuned checkpoint and compare the pretrained, shifted and fine-tuned small models with the uMLIPs.

In [ ]:
# You can change here------------------------------------------------------------
ft_path = ...   # path of the checkpoint you saved above
# -------------------------------------------------------------------------------

for label, path in [('7net-0-small', f'{DATA}/sevennet_0_small_checkpoint.pth'),
                    ('7net-0-small + shift', 'checkpoint_shift.pth'),
                    ('7net-0-small + FT', ft_path)]:
    calc = SevenNetCalculator(path)
    add_result(label, calc)
    free_gpu(calc)

summary = pd.DataFrame(results).T
summary.round(3)

In [ ]:
cols = ['Energy RMSE (eV/atom)', 'Force RMSE (eV/Å)', 'Stress RMSE (kbar)']
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, cols):
    summary[col].plot.bar(ax=ax, color=['#9DB4D3'] * (len(summary) - 1) + ['#2F5597'], logy=(col.startswith('Energy')))
    ax.set_title(col); ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.savefig('practice2_rmse.png', dpi=150); plt.show()

plot_parity(['7net-0-small', '7net-0-small + FT'])

**Check point.** (1) How much of the energy error does the shift alone remove? What does training add beyond the shift? (2) Compare the fine-tuned small model with the best uMLIP from Practice 1: which is better for forces, and which for stress?

## Practice 3. Relax the structure and run MD

Now use the model as a simulator, and look at the structures.

1. **Relaxation**: LBFGS optimizer + `FrechetCellFilter` (relaxes atoms and cell volume). Compare the volume per unit cell with experiment (956–960 Å³).
2. **Check against DFT**: compare each model's energy–volume curve with DFT.
3. **NVT MD**: Langevin dynamics at 600 K for 2 ps. Plot the mean squared displacement (MSD) of each element and the paths of the Li ions.

In [ ]:
from ase.optimize import LBFGS
from ase.filters import FrechetCellFilter
from ase.visualize.plot import plot_atoms
from ase.data import atomic_numbers
from ase.data.colors import jmol_colors

ELEMENTS = ['Li', 'P', 'S', 'Cl']
ATOM_COLORS = {el: jmol_colors[atomic_numbers[el]] for el in ELEMENTS}            # for atoms
LINE_COLORS = {'Li': '#A05EE0', 'P': '#E07000', 'S': '#C9A400', 'Cl': '#1A9E1A'}   # for lines

def relax(atoms, calc, fmax=0.05, steps=1000):
    """Relax atoms and cell. Returns the relaxed structure and the volume per unit cell at each step."""
    a = atoms.copy(); a.calc = calc
    opt = LBFGS(FrechetCellFilter(a, hydrostatic_strain=True), logfile=None)
    vol = []
    opt.attach(lambda: vol.append(a.get_volume() / 8))        # 2x2x2 supercell
    opt.run(fmax=fmax, steps=steps)
    return a, np.array(vol)

start = read(f'{DATA}/lpsc_test_md.extxyz.gz', 0)
volumes, traces = {}, {}
for label, path in [('7net-0', '7net-0'), ('7net-0-small + FT', ft_path)]:
    calc = SevenNetCalculator(path)
    relaxed, traces[label] = relax(start, calc)
    volumes[label] = traces[label][-1]
    print(f'{label:18s} {volumes[label]:7.1f} Å³ per unit cell ({len(traces[label]) - 1} steps)')
    if label == '7net-0-small + FT':
        relaxed_ft = relaxed
    free_gpu(calc)
print('Experiment         956–960 Å³ per unit cell')

In [ ]:
fig = plt.figure(figsize=(15, 4.8))
for k, (a, title) in enumerate([(start, 'Start: DFT MD snapshot'), (relaxed_ft, 'Relaxed (fine-tuned model)')]):
    ax = fig.add_subplot(1, 3, k + 1)
    plot_atoms(a, ax, radii=0.5, rotation='10x,10y')
    ax.set_title(f'{title}\n{a.get_volume() / 8:.1f} Å³ per unit cell'); ax.axis('off')
ax = fig.add_subplot(1, 3, 3)
for label, vol in traces.items():
    ax.plot(vol, label=label)
ax.axhspan(956, 960, color='#2F5597', alpha=0.15, label='Experiment')
ax.set_xlabel('Optimizer step'); ax.set_ylabel('Volume per unit cell (Å³)'); ax.legend()
handles = [plt.Line2D([], [], marker='o', ls='', mfc=ATOM_COLORS[e], mec='k', ms=9, label=e) for e in ELEMENTS]
fig.legend(handles=handles, loc='lower left', ncol=4, frameon=False, bbox_to_anchor=(0.12, 0))
plt.tight_layout(rect=(0, 0.07, 1, 1)); plt.savefig('practice3_relax.png', dpi=150); plt.show()

Rotate the relaxed structure with the mouse.

In [ ]:
from ase.visualize import view
view(relaxed_ft, viewer='x3d')

### Check against DFT: energy–volume curve

The relaxed volumes are above experiment. Is that the model's error or DFT's? Compare each model with a DFT energy–volume curve of Li₆PS₅Cl (11 volumes, from the SevenNet tutorial data). Energies are shown relative to each curve's minimum, so the energy offset does not matter here. This DFT curve uses Materials Project-compatible PBE settings, like the uMLIP training data; our fine-tuning data used different settings and only two cell volumes (961 and 965 Å³).

In [ ]:
from ase.eos import EquationOfState

eos_set = read(f'{DATA}/lpsc_eos.extxyz.gz', ':')
vols = np.array([a.get_volume() / 8 for a in eos_set])                 # per unit cell
curves = {'DFT': np.array([a.get_potential_energy() / len(a) for a in eos_set])}
EOS_MODELS = {
    '7net-0': ('7net-0', {}),
    '7net-omni': ('7net-omni', {'modal': 'mpa'}),
    '7net-0-small + FT': (ft_path, {}),
}
for label, (name, options) in EOS_MODELS.items():
    calc = SevenNetCalculator(name, **options)
    curves[label] = predict(calc, eos_set)[0]
    free_gpu(calc)

eos_v0 = {}
fig, ax = plt.subplots(figsize=(7, 4.8))
for label, E in curves.items():
    v0, e0, B = EquationOfState(vols, E, eos='birchmurnaghan').fit()
    eos_v0[label] = v0
    style = dict(color='k', lw=2.5, marker='o') if label == 'DFT' else dict(lw=1.5, marker='.')
    ax.plot(vols, (E - e0) * 1000, label=f'{label}: V0 = {v0:.0f} Å³', **style)
ax.axvspan(956, 960, color='#2F5597', alpha=0.15, label='Experiment 956–960 Å³')
ax.set_xlabel('Volume per unit cell (Å³)'); ax.set_ylabel('E − E_min (meV/atom)')
ax.legend(fontsize=9); ax.set_title('Energy–volume curve of Li₆PS₅Cl')
plt.tight_layout(); plt.savefig('practice3_eos.png', dpi=150); plt.show()

pd.DataFrame({'EOS minimum V0': pd.Series(eos_v0),
              'Relaxed (this notebook)': pd.Series(volumes)}).round(1)

In [ ]:
from ase import units
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary

def run_nvt(atoms, calc, T, n_steps, dt_fs=2.0, interval=10, seed=0):
    """Langevin NVT MD. Returns times (ps), positions (frames x atoms x 3), temperatures."""
    a = atoms.copy(); a.calc = calc
    MaxwellBoltzmannDistribution(a, temperature_K=T, rng=np.random.default_rng(seed))
    Stationary(a)
    dyn = Langevin(a, dt_fs * units.fs, temperature_K=T, friction=0.01 / units.fs,
                   rng=np.random.default_rng(seed + 1))
    pos, temp = [], []                 # the observer also runs at step 0
    def record():
        pos.append(a.get_positions().copy()); temp.append(a.get_temperature())
    dyn.attach(record, interval=interval)
    dyn.run(n_steps)
    t = np.arange(len(pos)) * interval * dt_fs / 1000
    return t, np.array(pos), np.array(temp)

def msd_by_element(pos, atoms):
    """MSD (Å²) of each element; the center-of-mass drift is removed."""
    m = atoms.get_masses()
    disp = pos - pos[0]
    disp -= (disp * m[None, :, None]).sum(1, keepdims=True) / m.sum()
    sym = np.array(atoms.get_chemical_symbols())
    return {el: (disp[:, sym == el] ** 2).sum(-1).mean(1) for el in ELEMENTS}

def plot_li_paths(ax, atoms, pos, title):
    """Top view of the bottom half of the cell: framework atoms at the start and Li paths."""
    sym = np.array(atoms.get_chemical_symbols())
    sel = pos[0, :, 2] < atoms.cell[2, 2] / 2
    for el in ['S', 'P', 'Cl']:
        m = (sym == el) & sel
        ax.scatter(pos[0, m, 0], pos[0, m, 1], s=40, color=ATOM_COLORS[el], edgecolors='k', linewidths=0.3, label=el)
    for i in np.where((sym == 'Li') & sel)[0]:
        ax.plot(pos[:, i, 0], pos[:, i, 1], color=LINE_COLORS['Li'], lw=0.8)
    ax.plot([], [], color=LINE_COLORS['Li'], lw=2, label='Li path')
    ax.set_aspect('equal'); ax.set_xlabel('x (Å)'); ax.set_ylabel('y (Å)'); ax.set_title(title)
    ax.legend(loc='upper right', fontsize=9)

MD_STEPS = 1000                      # 2 fs x 1000 = 2 ps
calc = SevenNetCalculator(ft_path)
t, pos, temp = run_nvt(relaxed_ft, calc, T=600, n_steps=MD_STEPS)
free_gpu(calc)
print(f'mean temperature {temp[len(temp)//5:].mean():.0f} K')

msd = msd_by_element(pos, relaxed_ft)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for el, y in msd.items():
    axes[0].plot(t, y, label=el, color=LINE_COLORS[el])
axes[0].set_xlabel('Time (ps)'); axes[0].set_ylabel('MSD (Å²)'); axes[0].legend(); axes[0].set_title('MSD, NVT 600 K')
plot_li_paths(axes[1], relaxed_ft, pos, 'Li paths over 2 ps (bottom half of the cell)')
plt.tight_layout(); plt.savefig('practice3_msd.png', dpi=150); plt.show()

*(Optional)* Watch the MD in 3D. Purple: Li, orange: P, yellow: S, green: Cl.

In [ ]:
!pip install -q py3Dmol
import io, py3Dmol
from ase import Atoms
from ase.io import write

frames = [Atoms(relaxed_ft.get_chemical_symbols(), positions=p) for p in pos[::5]]
buf = io.StringIO(); write(buf, frames, format='xyz')
v = py3Dmol.view(width=700, height=500)
v.addModelsAsFrames(buf.getvalue(), 'xyz')
v.setStyle({'sphere': {'scale': 0.3, 'colorscheme': 'Jmol'}})
v.zoomTo(); v.animate({'loop': 'forward', 'interval': 80}); v.show()

**Check point.** Which element moves? Do the Li ions hop between sites or only rattle? Is 2 ps long enough to get a diffusion coefficient? (Post-class assignment B.) Why are the relaxed volumes larger than experiment, and what does the DFT energy–volume curve say? Why does the fine-tuned model's curve look different?

## Submission

Run the next cell and submit the five files.

1. `inclass_results.csv`: RMSE table of all models (Practice 1 and 2)
2. `practice2_rmse.png`: RMSE comparison
3. `practice3_relax.png`: structures before and after relaxation, and the volume during relaxation
4. `practice3_eos.png`: energy–volume curves of the models and DFT
5. `practice3_msd.png`: MSD and Li paths

In [ ]:
summary.to_csv('inclass_results.csv')
pd.Series(volumes, name='Volume per unit cell (Å³)').round(1).to_csv('inclass_volumes.csv')
print(summary.round(3)); print(pd.Series(volumes).round(1))

try:
    from google.colab import files
    for f in ['inclass_results.csv', 'inclass_volumes.csv', 'practice2_rmse.png', 'practice3_relax.png', 'practice3_eos.png', 'practice3_msd.png']:
        files.download(f)
except ImportError:
    pass